# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task Type:** Ranking / Scoring

**Why:** My lane is Refresh / Content Opportunity Scoring – the goal is to produce a ranked queue of pages for a content editor to review. This is fundamentally a **ranking** problem: we want the most promising pages at the top of the list.

I'm choosing **scoring** as the mechanism – each page gets a score, and we sort by that score descending. This is more flexible than classification because:
- It produces an ordered list that matches how an editor actually works
- It allows the editor to choose how many pages to review (top 20, top 50, etc.)
- It's easier to evaluate with Precision@K metrics

**Why not classification?** Classification would give a binary "review / don't review" label for each page. But that doesn't match the real workflow – editors don't have a fixed cutoff; they want a prioritized list. Ranking gives them that flexibility.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target / Proxy:** `is_declining_label` (a proxy label derived from `trend_direction`)

**Where does this label come from?**

The label comes from an **observed outcome** in the data – specifically, a page's impression trend over time:

- `is_declining_label = 1` when `trend_direction == "down"`
- `is_declining_label = 0` otherwise

The `trend_direction` is calculated from actual impression data:
- `trend_direction == "down"` when impressions in the last 30 days are **at least 20% lower** than impressions in the previous 30 days

So the label is based on **real observed behavior** – pages that actually lost traffic over time.

---

**Is this a perfect target? No – it's a proxy.**

A stronger target would be a **future-looking** outcome, like:
- "Will this page decline in the next 30 days?"
- Features from prior 90 days → label from next 30 days

But for this starter work, `is_declining_label` is a reasonable proxy because:
- It's based on observed data, not a hand-written rule
- It captures the idea of "pages that are losing traction"
- The starter pipeline already uses it and shows strong results

---

**The important distinction:**

| Good | Not Good |
|---|---|
| Label from observed outcome (`is_declining_label`) | Label from a human-defined rule |
| Label from future behavior (next 30 days) | Label from the same time period as features |
| Label we can defend with evidence | Label we can't explain or justify |

For this project, we're using the starter label as a **proxy** – it's not perfect, but it's a reasonable starting point that we can improve in later weeks.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary Success Metric:** Precision@K (specifically Precision@20 and Precision@50)

---

**What is Precision@K?**

Precision@K measures: *"Of the top K pages the system recommends, how many are actually worth reviewing?"*

Formula:
```
Precision@K = (Number of relevant pages in top K) / K
```

**Example:** If Precision@50 = 0.740, then 37 of the top 50 recommended pages are actually declining (relevant) – and 13 are false positives.

---

**Why this metric?**

| Criterion | Why Precision@K fits |
|---|---|
| **Matches the decision** | An editor reviews pages in order, stopping when time runs out. They don't care about the whole ranking equally – they care about the top of the list. |
| **Actionable** | If the team has capacity for 20 reviews, Precision@20 tells them how many of those 20 will be worth their time. |
| **Simple to communicate** | "37 of the top 50 pages are actually declining" is understandable to non-technical stakeholders. |
| **Honest** | It doesn't hide behind global accuracy – it measures what actually matters for the real workflow. |

---

**What about other metrics?**

| Metric | Why we use it | Why we don't use it alone |
|---|---|---|
| **Precision@K** | Primary metric – measures quality at the top of the queue | Doesn't measure recall (what we miss) |
| **Recall@K** | Secondary – measures how many total declining pages we catch | Less important than precision for this problem; editors care more about wasted time than missed opportunities |
| **ROC-AUC** | Useful for model comparison | Doesn't match the decision workflow – it measures overall ranking quality, not top-K performance |
| **Average Precision** | Useful for comparing models | Harder to communicate; Precision@K is more intuitive |
| **Accuracy** | Simple but misleading | The label is imbalanced (54% declining); accuracy would be inflated and meaningless |

---

**What number means "good"?**

| Threshold | Meaning |
|---|---|
| **Precision@50 ≥ 0.50** | At least 25 of the top 50 pages are worth reviewing – the editor's time is well spent |
| **Precision@50 ≥ 0.70** | 35+ of the top 50 are worth reviewing – strong performance |
| **Precision@50 ≥ 0.80** | 40+ of the top 50 are worth reviewing – excellent performance |

**In the starter pipeline:**
- Baseline (hand-written rule): Precision@50 = 0.240 (~12 of 50)
- Random Forest: Precision@50 = 0.740 (~37 of 50)

The random forest already exceeds the "good" threshold (≥ 0.70). Our goal is to maintain or improve this as we move to the full warehouse data.

---

**The trade-off we accept:**

Precision@K is **not** the same as "catching every declining page." We are optimizing for **efficient use of editor time**, not for perfect recall. This is a deliberate choice based on the cost analysis in Section 2: false positives are less expensive than false negatives, but we still want to minimize wasted time.

> **We are building a decision-support tool, not a perfect classifier. Precision@K matches that purpose.**

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd
import os, sys, subprocess

# Make sure we're in the right directory
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = "flyrank-ml-internship-starter"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1",
                       "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
                       REPO_DIR], check=True)
    os.chdir(REPO_DIR)

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("=" * 60)
print("UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (PAGE)")
print("=" * 60)
print(f"Total rows: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique clients: {df['client_id'].nunique():,}")
print("\n" + "=" * 60)
print("SAMPLE OF THE DATA (first 5 rows):")
print("=" * 60)

# Select key columns to show the unit of analysis clearly
key_columns = [
    'content_id',
    'client_id',
    'impressions_90d',
    'clicks_90d',
    'sessions_90d',
    'ctr',
    'avg_position',
    'content_age_days',
    'trend_direction'
]

display_df = df[key_columns].head(5)
display_df

UNIT OF ANALYSIS: ONE ROW = ONE CONTENT ITEM (PAGE)
Total rows: 30,000
Total columns: 44
Unique content items: 30,000
Unique clients: 32

SAMPLE OF THE DATA (first 5 rows):


,content_id,client_id,impressions_90d,clicks_90d,sessions_90d,ctr,avg_position,content_age_days,trend_direction
0,content_304f48230142,client_f369cb89fc,3803,29,17,0.76,10.6,187,down
1,content_a1fb4e703a9e,client_4e07408562,15320,7,9,0.05,20.3,445,down
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,11,0.09,36.5,141,down
3,content_331d6c4de07b,client_19581e27de,11751,58,78,0.49,6.2,463,stable
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,145,0.13,44.0,263,down


### What One Row Represents

**One row = one content item (page).**

Each row in this dataset represents a **single page** on a client's website. The data is aggregated over a trailing 90-day window, meaning all metrics (impressions, clicks, sessions, CTR, position, etc.) are measured over the same 90-day period.

This is the **unit of analysis** for our ranking problem:
- Each page gets a **score**
- Pages are **ranked** by that score
- The editor reviews pages in that order

---

**Key columns in the sample:**

| Column | What it tells us |
|---|---|
| `content_id` | Unique identifier for the page (pseudonymized) |
| `client_id` | Which client the page belongs to (pseudonymized) |
| `impressions_90d` | How many times the page appeared in search results |
| `clicks_90d` | How many times users clicked through |
| `sessions_90d` | How many GA4 sessions the page received |
| `ctr` | Click-through rate (×100 percentage – 0.35 = 0.35%) |
| `avg_position` | Average search result position (0 = no data) |
| `content_age_days` | How old the page is (in days) |
| `trend_direction` | Whether impressions are up, down, flat, stable, or new |

---

**Why this matters for our ranking problem:**

We are scoring and ranking **pages** – not clients, not days, not queries. Every row is a candidate for review. The ranking algorithm assigns a score to each page, and the editor reviews pages from highest score to lowest.

> **Unit of analysis: one content page. Decision: which pages to review first.**

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Why ML beats a fixed rule here**

The starter pipeline already gives us a direct answer: the learned model (random forest) achieves Precision@50 of 0.740, while the hand-written rule baseline achieves only 0.240.

But let's break down **why** – what makes this problem too messy for a simple if-statement?

---

### 1. Many signals interact in complex ways

A fixed rule might say:
- "If impressions > 500 and CTR < 0.5%, flag for review."
- "If position > 10 and impressions are dropping, flag for review."

But in reality, these signals interact:
- A page with high impressions but low CTR might still be fine if it's brand new
- A page with declining impressions might be seasonal, not a real problem
- A page with good CTR but falling position might need different action

A hand-written rule can't capture all these interactions. An ML model can learn them from data.

---

### 2. The relationship is non-linear

The relationship between signals and "worth reviewing" is not a straight line:
- More impressions = more important, but only up to a point
- Lower position = worse, but pages at position 1 with low CTR are more suspicious than pages at position 20 with low CTR
- Older content might need refreshing, but some old content is evergreen and performs well

ML models (especially tree-based models like random forest) naturally handle non-linear relationships without us having to define them.

---

### 3. The threshold is not obvious

A fixed rule requires a human to choose thresholds:
- "At least 500 impressions" – but why 500? Why not 300? Or 1000?
- "CTR below 0.5%" – but why 0.5%? What about pages where CTR is low because of low impressions?

An ML model learns the optimal thresholds from the data, without us having to guess.

---

### 4. The pattern changes over time

The relationship between signals and "worth reviewing" may change as search algorithms change, as user behavior changes, or as content ages.

A fixed rule would need to be updated manually every time the pattern shifts. An ML model can be retrained on fresh data to adapt to new patterns.

---

### 5. The evidence is in the numbers

From the starter pipeline:

| Method | Precision@50 | Improvement |
|---|---|---|
| Hand-written rule (baseline) | 0.240 | — |
| Logistic regression | 0.400 | +67% |
| Decision tree | 0.540 | +125% |
| Random forest | 0.740 | +208% |

Each step toward a more flexible, data-driven approach improves performance. This is not a coincidence – it's evidence that the pattern is too complex for a simple rule.

---

### The Bottom Line

> **A fixed rule is like a chef following a recipe. An ML model is like a chef who tastes and adjusts.**
>
> When the relationship between inputs and outputs is messy, shifting, and non-linear, ML can learn patterns that no human could write down as a set of if-statements.

For this problem:
- Signals interact
- Relationships are non-linear
- Thresholds are not obvious
- Patterns evolve over time

These four factors make this a genuine ML problem – not just a problem that could be solved with a few hand-written rules.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.